## **Sección 1**: *Simulated Annealing* (Recocido Simulado)

### 1.1 Implementación del algoritmo

> Implemente el algoritmo de **Simulated Annealing (Recocido Simulado)** empleando el **schedule geométrico** como esquema para controlar la evolución de la temperatura $T$ a lo largo de las iteraciones. Emplee como criterio de parada un número máximo de iteraciones $N_{\text{max}}$, y considere además los siguientes criterios complementarios:
> - **Temperatura mínima**: detener cuando $T < T_{\text{min}}$ (ej: $T_{\text{min}} = 10^{-3}$)
> - **Estancamiento**: detener si no hay mejora en el mejor fitness durante $k$ iteraciones consecutivas
> - **Número de evaluaciones**: detener después de un número fijo de evaluaciones de la función objetivo (relevante para la comparación solicitada en la Sección 3)
> Puede encontrar más detalles del algoritmo en el **Anexo I**.


> Analice el funcionamiento del algoritmo durante la minimización de la función $f_1(x)$ de Schwefel, empleando los siguientes parámetros iniciales: $T_0 = 100$, $\alpha = 0.95$, $N_{\text{max}} = 1000$, inicialización aleatoria en $[-512, 512]$. Ejecute el algoritmo varias veces y realice las siguientes actividades: (1) grafique la evolución del fitness y determine si presenta oscilaciones, y si disminuyen con el descenso de la temperatura, (2) si el algoritmo logra escapar de óptimos locales en etapas tempranas de la exploración, (3) en qué momento de la ejecución típicamente se encuentra la mejor solución. Experimente modificando $T_0$ y $\alpha$ para desarrollar intuición sobre su efecto en el comportamiento del algoritmo.


### 1.2 Análisis de hiperparámetros en Schwefel 1D

Los dos hiperparámetros fundamentales de *Simulated Annealing* con schedule geométrico son la temperatura inicial $T_0$ y la tasa de enfriamiento $\alpha$. Estos parámetros controlan el balance entre exploración y explotación a lo largo de la ejecución del algoritmo. Una calibración sistemática de estos hiperparámetros es esencial para maximizar el rendimiento del algoritmo en problemas específicos.


#### Experimento 1: Temperatura inicial

> Evalúe el efecto del valor de la temperatura inicial sobre la capacidad del algoritmo para encontrar el óptimo global de la función de Schwefel, manteniendo constante la tasa de enfriamiento.



**Configuración experimental**:

$$
\begin{aligned}
\text{Función:} & \quad f_1(x) = -x \cdot \sin(\sqrt{|x|}), \quad x \in [-512, 512] \\
\text{Hiperparámetros:} & \quad T_0 \in \{10, 50, 100, 500, 1000\} \\
& \quad \alpha = 0.95 \text{ (fijo)} \\
\text{Iteraciones:} & \quad N_{\text{max}} = 1000 \\
\text{Repeticiones:} & \quad 10 \text{ ejecuciones independientes por cada } T_0 \\
\text{Inicialización:} & \quad \text{Aleatoria uniforme en } [-512, 512] \\
\text{Vecindario:} & \quad x' = x + \mathcal{N}(0, \sigma^2), \quad \sigma = 50
\end{aligned}
$$


> Para cada una de las 10 repeticiones de cada valor de $T_0$, se registran las siguientes métricas:
> - **Mejor fitness alcanzado**: $f(x_{\text{best}})$
> - **Posición de la mejor solución**: $x_{\text{best}}$
> - **Iteración en que se encontró**: $t_{\text{best}}$
> - **Número de aceptaciones de movimientos que empeoran**: para calcular tasa de aceptación
> Para cada valor de $T_0 \in \{10, 50, 100, 500, 1000\}$, calcular sobre las 10 repeticiones:
> 1. **Media aritmética** del mejor fitness: $\bar{f} = \frac{1}{10}\sum_{i=1}^{10} f_i$
> 2. **Desviación estándar**: $s = \sqrt{\frac{1}{9}\sum_{i=1}^{10}(f_i - \bar{f})^2}$, que cuantifica la variabilidad del algoritmo
> 3. **Mediana**: valor central que divide la distribución en dos mitades iguales, más robusta que la media ante outliers
> 4. **Valores extremos**: mínimo y máximo observados en las 10 repeticiones
> Organizar estos resultados en una tabla comparativa. Adicionalmente, genere un boxplot por cada valor de $T_0$ mostrando la distribución de los mejores fitness alcanzados.

| $T_0$ | Media | Mediana | Mín | Máx | Std |
|:-----:|:-----:|:-------:|:---:|:---:|:---:|
| 10    | ...   | ...     | ... | ... | ... |
| 50    | ...   | ...     | ... | ... | ... |
| ...   | ...   | ...     | ... | ... | ... |



**Preguntas de análisis**:

1. ¿Qué valor de $T_0$ produce el mejor rendimiento promedio? ¿Es el mismo que produce el mejor resultado individual (mínimo absoluto)?

2. ¿Cómo afecta $T_0$ a la variabilidad de los resultados (desviación estándar)? ¿Configuraciones con $T_0$ alto son más o menos consistentes?

3. ¿Temperaturas muy altas ($T_0 = 1000$) mejoran o perjudican el rendimiento? ¿Por qué?

4. ¿Cuántas de las 10 repeticiones de cada configuración logran encontrar el óptimo global (dentro de un $\varepsilon = 5$ de tolerancia)?


#### Experimento 2: Tasa de enfriamiento


Una vez identificado un valor apropiado de temperatura inicial en el Experimento 1, se procede a evaluar el impacto de la tasa de enfriamiento $\alpha$ sobre el rendimiento del algoritmo.

**Configuración experimental**:

$$
\begin{aligned}
\text{Función:} & \quad f_1(x), \quad x \in [-512, 512] \\
\text{Hiperparámetros:} & \quad T_0 = [T_{0,\text{óptimo}} \text{ del Exp.1}] \text{ (fijo)} \\
& \quad \alpha \in \{0.80, 0.85, 0.90, 0.95, 0.99\} \\
\text{Protocolo:} & \quad \text{Idéntico al Experimento 1}
\end{aligned}
$$



El parámetro $\alpha$ controla la velocidad del enfriamiento: valores cercanos a 1 producen enfriamiento lento que permite exploración prolongada pero requiere más iteraciones para converger, mientras que valores más pequeños producen enfriamiento rápido que favorece convergencia temprana pero con mayor riesgo de estancamiento en óptimos locales. Para el mismo número de iteraciones $N_{\text{max}} = 1000$, diferentes valores de $\alpha$ resultan en temperaturas finales muy diferentes:

- $\alpha = 0.80$: $T_{\text{final}} = T_0 \cdot 0.80^{1000} \approx 0$ (enfriamiento muy rápido)
- $\alpha = 0.95$: $T_{\text{final}} = T_0 \cdot 0.95^{1000} \approx 5.3 \times 10^{-23}$ (enfriamiento moderado)
- $\alpha = 0.99$: $T_{\text{final}} = T_0 \cdot 0.99^{1000} \approx 4.3 \times 10^{-5} T_0$ (enfriamiento lento)


> Aplicar el mismo protocolo de análisis que en el Experimento 1, generando tabla comparativa y visualizaciones. Adicionalmente, para cada valor de $\alpha$, calcular y comparar:
> - **Tasa de aceptación promedio**: proporción de movimientos que empeoran el fitness y son aceptados
> - **Iteración promedio del mejor hallazgo**: $\bar{t}_{\text{best}}$, indica cuándo típicamente se encuentra la mejor solución


**Preguntas de análisis**:

1. ¿Qué valor de $\alpha$ produce el mejor balance entre exploración y explotación para esta función?

2. ¿Existe correlación entre la tasa de aceptación y la calidad de las soluciones encontradas?

3. ¿El mejor $\alpha$ es el mismo si se mide por rendimiento promedio vs por mejor caso?

4. ¿Enfriamiento lento ($\alpha = 0.99$) siempre es mejor dado un número fijo de iteraciones?

5. ¿Cómo interactúa $\alpha$ con $T_0$? ¿La configuración óptima de uno depende del otro?


#### Experimento 3: Robustez ante inicialización



Los experimentos anteriores utilizan inicialización aleatoria uniforme en todo el dominio. Este experimento evalúa la robustez del algoritmo, con su configuración óptima identificada, ante diferentes estrategias de inicialización, particularmente inicialización en regiones lejanas del óptimo global.



**Configuración experimental**:

$$
\begin{aligned}
\text{Hiperparámetros:} & \quad T_0 = [T_{0,\text{óptimo}}], \quad \alpha = [\alpha_{\text{óptimo}}] \\
\text{Estrategias de inicialización:} & \\
& \quad \text{(a) Aleatoria uniforme en } [-512, 512] \text{ (baseline)} \\
& \quad \text{(b) Región cercana al origen: } [-50, 50] \\
& \quad \text{(c) Región lejana (negativa): } [-512, -200] \\
& \quad \text{(d) Región lejana (positiva, lado incorrecto): } [50, 250] \\
\text{Repeticiones:} & \quad 10 \text{ por cada estrategia}
\end{aligned}
$$


>Comparar las cuatro estrategias en términos de:
> - Capacidad de alcanzar el óptimo global ($x^* \approx 420.97$)
> - Variabilidad entre repeticiones
> - Número de iteraciones necesarias para convergencia



**Preguntas**:

1. ¿La configuración óptima es robusta ante diferentes inicializaciones?
2. ¿Inicializar cerca del origen (región engañosa) perjudica significativamente el rendimiento?
3. ¿Qué proporción de ejecuciones logran escapar de sus cuencas iniciales y alcanzar el óptimo global?


### 1.3 Aplicación a función 2D: Oscilación Radial


#### Adaptaciones necesarias para 2D



**Operador de vecindario bidimensional**: La generación de vecinos en $\mathbb{R}^2$ se realiza mediante perturbación Gaussiana independiente en cada dimensión:

$$\begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} x \\ y \end{pmatrix} + \begin{pmatrix} \delta_x \\ \delta_y \end{pmatrix}, \quad \delta_x, \delta_y \sim \mathcal{N}(0, \sigma^2)$$

donde $\sigma$ es la desviación estándar que controla el tamaño del paso de búsqueda. Un valor apropiado inicial para la función de Oscilación Radial con dominio $[-100, 100]^2$ es $\sigma = 10.0$, que representa aproximadamente el 10% del rango del dominio, permitiendo exploración efectiva sin pasos excesivamente grandes.

**Manejo de restricciones del dominio**: Cuando la perturbación genera un candidato $(x', y')$ que viola las restricciones del dominio, se debe aplicar una estrategia de corrección. Para este trabajo práctico se empleará la estrategia de **proyección al borde**: si $x' < -100$ se fija $x' = -100$, si $x' > 100$ se fija $x' = 100$, y análogamente para $y'$. Esto garantiza que todos los candidatos evaluados sean factibles sin desperdiciar evaluaciones.



#### Experimento 1: Calibración de temperatura inicial en 2D


El primer paso es determinar si la temperatura inicial $T_0$ óptima identificada en Schwefel 1D es apropiada para la función de Oscilación Radial, o si requiere ajuste debido a las diferencias en la escala y estructura del paisaje de fitness.



**Configuración experimental**:

$$
\begin{aligned}
\text{Función:} & \quad f_2(x,y) = (x^{2} + y^{2})^{0.25} \cdot [\sin^{2}(50 \cdot (x^{2} + y^{2})^{0.1}) + 1] \\
\text{Dominio:} & \quad (x,y) \in [-100, 100]^2 \\
\text{Hiperparámetros:} & \quad T_0 \in \{50, 100, 200, 500\} \\
& \quad \alpha = [\alpha_{\text{óptimo}} \text{ de Exp. 1.3.2}] \text{ (fijo)} \\
& \quad \sigma = 10.0 \text{ (desviación estándar de perturbación)} \\
\text{Iteraciones:} & \quad N_{\text{max}} = 2000 \\
\text{Repeticiones:} & \quad 10 \text{ ejecuciones independientes por cada } T_0 \\
\text{Inicialización:} & \quad (x_0, y_0) \text{ aleatorio uniforme en } [-100, 100]^2
\end{aligned}
$$



**Justificación de valores de $T_0$**: El rango de valores de $f_2$ en el dominio es significativamente diferente al de Schwefel 1D. Evaluaciones preliminares muestran que $f_2$ típicamente varía en el rango $[0, 100]$ aproximadamente, con incrementos $\Delta f$ que pueden ser del orden de 1-10 en pasos locales. Por lo tanto, se evalúan temperaturas desde 50 (conservador) hasta 500 (muy exploratorio).



**Métricas a registrar** (para cada una de las 10 repeticiones de cada configuración):

1. **Mejor fitness alcanzado**: $f_2(x_{\text{best}}, y_{\text{best}})$
2. **Posición de la mejor solución**: $(x_{\text{best}}, y_{\text{best}})$
3. **Distancia euclidiana al óptimo global**: $d = \sqrt{x_{\text{best}}^2 + y_{\text{best}}^2}$
4. **Iteración en que se encontró el mejor**: $t_{\text{best}}$
5. **Tasa de aceptación acumulada**: proporción de movimientos que empeoran y fueron aceptados


> Para cada valor de $T_0$, calcular sobre las 10 repeticiones:
> - Media aritmética de $f_{\text{best}}$
> - Mediana, mínimo y máximo de $f_{\text{best}}$
> - Desviación estándar
> - Media de la distancia al óptimo $\bar{d}$ 
> - Tiempo promedio de ejecución
> Presentar los resultados en una **tabla comparativa**:


| $T_0$ | Media | Mediana | Mín | Máx | Dist. promedio |
|-------|-------|---------|-----|-----|----------------|
| 50    | ...   | ...     | ... | ... | ...            |
| 100   | ...   | ...     | ... | ... | ...            |
| 200   | ...   | ...     | ... | ... | ...            |
| 500   | ...   | ...     | ... | ... | ...            |


> Adicionalmente, generar visualizaciones complementarias:
> 1. **Boxplot comparativo**: Un gráfico de boxplot que muestre, para cada valor de $T_0$, la distribución de los mejores fitness alcanzados.
> 2. **Scatter plot de posiciones finales**: Sobre un contour plot de la función, marcar las 10 posiciones finales $(x_{\text{best}}, y_{\text{best}})$ de cada configuración con diferentes colores. Incluir un círculo en el origen y círculos concéntricos a distancias 10, 25, 50 como referencia visual.
> 3. **Gráfico de convergencia promedio**: Para cada $T_0$, graficar la evolución del fitness promedio (promediado sobre las 10 repeticiones) vs iteración.

**Preguntas de análisis**:

1. ¿Qué valor de $T_0$ produce el mejor rendimiento promedio en términos de fitness alcanzado?
2. ¿La temperatura óptima en 2D coincide con la identificada en Schwefel 1D, o requiere ajuste?
3. ¿Cómo afecta $T_0$ a la distancia promedio al óptimo global? ¿Mayor temperatura facilita acercarse al origen?
4. ¿Existe una correlación entre la tasa de aceptación y la calidad de las soluciones finales?
5. ¿Cuántas de las 10 repeticiones de cada configuración logran encontrar soluciones dentro de un radio $d < 10$ del óptimo?


#### Experimento 2: Impacto de la desviación estándar del operador de vecindario


El parámetro $\sigma$ que controla el tamaño de los pasos en el operador de vecindario es crítico en problemas continuos. Pasos muy pequeños resultan en exploración local lenta, mientras que pasos muy grandes pueden generar movimientos erráticos que dificultan la convergencia fina hacia el óptimo.


**Configuración experimental**:

$$
\begin{aligned}
\text{Hiperparámetros:} & \quad T_0 = [T_{0,\text{óptimo}} \text{ del Exp. 1}] \text{ (fijo)} \\
& \quad \alpha = [\alpha_{\text{óptimo}}] \text{ (fijo)} \\
& \quad \sigma \in \{2.0, 5.0, 10.0, 20.0, 40.0\} \\
\text{Protocolo:} & \quad \text{Idéntico al Experimento 1}
\end{aligned}
$$

**Justificación del rango de $\sigma$**:
- $\sigma = 2.0$: pasos pequeños (~2% del rango del dominio), exploración local fina
- $\sigma = 10.0$: pasos moderados (~10% del rango), balance exploración-explotación
- $\sigma = 40.0$: pasos grandes (~40% del rango), exploración global amplia


> Aplicar el mismo protocolo estadístico del Experimento 1. Adicionalmente, para cada valor de $\sigma$, calcular:
> - **Número promedio de mejoras**: cuántas veces se actualiza $f_{\text{best}}$ a lo largo de las 2000 iteraciones
> - **Magnitud promedio de mejora**: cuando ocurre una mejora, cuál es el $|\Delta f|$ típico



**Preguntas de análisis**:

1. ¿Qué valor de $\sigma$ produce el mejor balance entre exploración global y refinamiento local?
2. ¿Pasos muy pequeños ($\sigma = 2.0$) resultan en estancamiento por exploración insuficiente del espacio?
3. ¿Pasos muy grandes ($\sigma = 40.0$) impiden convergencia fina al óptimo?
4. ¿Existe interacción entre $\sigma$ y $T_0$? ¿Temperaturas altas requieren pasos diferentes que temperaturas bajas?
5. Observando el gráfico de contorno con las trayectorias típicas, ¿cómo difiere el patrón de búsqueda para $\sigma$ pequeño vs grande?


#### Análisis integrador

Una vez completados los tres experimentos, se debe realizar una síntesis que responda:

1. **Transferencia de hiperparámetros**: ¿Los valores óptimos de $T_0$ y $\alpha$ identificados en Schwefel 1D transfieren efectivamente a Oscilación Radial 2D, o requieren ajuste sustancial?

2. **Rol del operador de vecindario**: ¿El parámetro $\sigma$ tiene un impacto comparable o mayor que $T_0$ y $\alpha$ en el rendimiento 2D?

3. **Desafíos de dimensionalidad**: ¿La función 2D presenta desafíos cualitativamente diferentes a la 1D que afectan el comportamiento de SA? Por ejemplo, ¿la convergencia es más lenta? ¿La variabilidad entre repeticiones es mayor?

4. **Visualización del comportamiento**: Basándose en las visualizaciones en tiempo real ejecutadas durante los experimentos, describir cualitativamente cómo SA navega el paisaje radial con oscilaciones. ¿Sigue una trayectoria predominantemente radial hacia el origen, o exhibe movimientos más erráticos? ¿A qué se debe?

---